In [1]:
import pandas as pd
from unidecode import unidecode
import geopandas as gpd
import pydeck as pdk
import json
import webbrowser
from pathlib import Path
from datetime import datetime
import os
from functools import reduce
import duckdb
import numpy as np


usuario = os.getlogin()
current_date = datetime.now().strftime("%d-%m-%y")

In [2]:
base_tode = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos/Productividad/conteos con ece/todes_consolidadas.parquet")
base_metas = pd.read_excel(rf"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos\Productividad\Metas\2026\Metas de productividad por unidad medica 2026.xlsx")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")


In [3]:
query = """
WITH base AS (
    SELECT
        clues AS clues_imb,

        SUM(CASE WHEN tipo_consulta = 'general' THEN procedimientos ELSE 0 END) AS general,
        SUM(CASE WHEN tipo_consulta = 'qx' THEN procedimientos ELSE 0 END) AS qx,
        SUM(CASE WHEN tipo_consulta = 'especialidad' THEN procedimientos ELSE 0 END) AS especialidad,
        SUM(CASE WHEN tipo_consulta = 'egresos' THEN procedimientos ELSE 0 END) AS egresos

    FROM base_tode
    WHERE anio_insert = 2026
    GROUP BY clues
),

metas AS (
    SELECT
        entidad,
        clues_imb,
        estatus_de_operacion,
        nombre_de_la_unidad,
        nivel_atencion,
        meta_general_anual,
        meta_especialidad_anual,
        meta_cirugia_anual,
        meta_egresos_anual
    FROM base_metas
)

SELECT
    m.clues_imb,
    m.entidad,
    m.estatus_de_operacion,
    m.nombre_de_la_unidad,
    m.nivel_atencion,

    b.general,
    b.qx,
    b.especialidad,
    b.egresos,

    m.meta_general_anual,
    m.meta_especialidad_anual,
    m.meta_cirugia_anual,
    m.meta_egresos_anual

FROM metas m
LEFT JOIN base b
    ON m.clues_imb = b.clues_imb
"""
base = duckdb.query(query).to_df()
base = base.rename(columns={
    "qx": "cirugias",})

In [4]:
cols = [
    'clues_imb', 'entidad', 'estatus_de_operacion', 'nombre_de_la_unidad',
    'nivel_atencion', 'general', 'cirugias', 'especialidad', 'egresos',
    'meta_general_anual', 'meta_especialidad_anual',
    'meta_cirugia_anual', 'meta_egresos_anual'
]

base = base[cols].copy()

#  consultas según nivel de atención
base['consultas'] = np.where(
    base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL']),
    base['cirugias'],
    base[['general', 'cirugias', 'especialidad', 'egresos']].sum(axis=1)
)

#  meta total (siempre suma de todas las metas)
base['meta_total'] = (
    base['meta_general_anual']
    + base['meta_especialidad_anual']
    + base['meta_cirugia_anual']
    + base['meta_egresos_anual']
)

In [5]:
cols_base =['clues_imb', 'entidad', 'nombre_de_la_unidad',
       'nivel_atencion', 'cirugias','meta_cirugia_anual','consultas', 'meta_total']
base = base[cols_base].copy()   

In [6]:
base = base.merge(
    clues[['clues_imb', 'latitud', 'longitud']], 
    on='clues_imb',
    how='left'
)

In [7]:
columnas_finales = [
 'clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud'
]
base = base.drop_duplicates(subset=columnas_finales, keep="first")

In [8]:
base.columns

Index(['clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud'],
      dtype='object')

In [11]:
# ============================================================
# 1) IMPORTAR LIBRERIAS
# ============================================================
import pandas as pd
import pydeck as pdk
import webbrowser
from pathlib import Path

# ============================================================
# 2) DEFINIR FUNCIONES AUXILIARES
# ============================================================

def asignar_macro_categoria(nivel_atencion):
    """
    Asigna macro categoria basada en nivel de atencion
    - SEGUNDO y TERCER NIVEL: Cirugias
    - Otros: Consultas
    """
    if nivel_atencion in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        return "cirugias"
    else:
        return "consultas"

# Configuracion de colores para el semaforo de avance
semaforo_colores = {
    "rojo": "#D41111",
    "amarillo": "#F1D54A",
    "verde claro": "#88A91E",
    "verde fuerte": "#0D5D2A"
}

def obtener_color_semaforo(porcentaje):
    """Devuelve el color hex y RGB para el semaforo basado en el porcentaje de avance"""
    if porcentaje < 50:
        return semaforo_colores["rojo"]
    elif porcentaje < 80:
        return semaforo_colores["amarillo"]
    elif porcentaje < 100:
        return semaforo_colores["verde claro"]
    else:
        return semaforo_colores["verde fuerte"]

def hex_to_rgb(hex_color, opacity=200):
    """Convierte color hex a RGB con opacidad especificada"""
    hex_color = hex_color.replace("#", "")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return [r, g, b, opacity]

# ============================================================
# 3) PREPARACION DE DATOS PARA EL MAPA
# ============================================================

# ¡ASEGURATE DE QUE base ESTE CARGADA!
# Si no lo esta, descomenta y ajusta:
# base = pd.read_csv("ruta_de_tu_archivo.csv")

df_plot = base.copy()

# Limpieza de datos
df_plot["latitud"] = pd.to_numeric(df_plot["latitud"], errors="coerce")
df_plot["longitud"] = pd.to_numeric(df_plot["longitud"], errors="coerce")

# Calcular metricas segun nivel de atencion
def calcular_indicador(row):
    """Calcula el indicador relevante segun nivel de atencion"""
    if row["nivel_atencion"] in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        numerador = row.get("cirugias", 0)
        denominador = row.get("meta_cirugia_anual", 1)
        avance = (numerador / denominador * 100) if denominador > 0 else 0
        indicador = "cirugias"
    else:
        numerador = row.get("consultas", 0)
        denominador = row.get("meta_total", 1)
        avance = (numerador / denominador * 100) if denominador > 0 else 0
        indicador = "consultas"
    
    return pd.Series({
        "numerador": numerador,
        "denominador": denominador,
        "pct_avance_meta_general": avance,
        "tipo_indicador": indicador
    })

# Aplicar calculo de indicadores
df_plot[["numerador", "denominador", "pct_avance_meta_general", "tipo_indicador"]] = df_plot.apply(calcular_indicador, axis=1)

# Limpiar y filtrar datos validos
df_plot = df_plot.dropna(subset=["latitud", "longitud", "pct_avance_meta_general"]).copy()
df_plot = df_plot[df_plot["denominador"] > 0].copy()

# Asignar macro categoria (para agrupar en estadisticas)
df_plot["macro_key"] = df_plot["nivel_atencion"].apply(asignar_macro_categoria)

# Asignar colores de las barras basado en el semaforo de avance
df_plot["color_hex"] = df_plot["pct_avance_meta_general"].apply(obtener_color_semaforo)
df_plot["color_rgb"] = df_plot["color_hex"].apply(lambda x: hex_to_rgb(x))

# Volumen para altura de columnas
df_plot["volumen_para_columna"] = df_plot["numerador"]

# Formatear para tooltip
df_plot["numerador_fmt"] = df_plot["numerador"].map(lambda x: f"{int(x):,}")
df_plot["denominador_fmt"] = df_plot["denominador"].map(lambda x: f"{int(x):,}")
df_plot["pct_fmt"] = df_plot["pct_avance_meta_general"].map(lambda x: f"{x:.1f}%")

# Funcion para obtener el nombre del color del semaforo para el tooltip
def obtener_estado_semaforo(pct):
    if pct < 50:
        return "Rojo (Muy Bajo)"
    elif pct < 80:
        return "Amarillo (Bajo)"
    elif pct < 100:
        return "Verde Claro (Cerca de Meta)"
    else:
        return "Verde Fuerte (Meta Superada)"

df_plot["estado_semaforo"] = df_plot["pct_avance_meta_general"].apply(obtener_estado_semaforo)

# ============================================================
# 4) CALCULAR ESTADISTICAS
# ============================================================

unidades_cirugias = len(df_plot[df_plot["macro_key"] == "cirugias"])
unidades_consultas = len(df_plot[df_plot["macro_key"] == "consultas"])

# Calcular totales con validacion
if unidades_cirugias > 0:
    total_cirugias = df_plot[df_plot["macro_key"] == "cirugias"]["numerador"].sum()
    meta_cirugias = df_plot[df_plot["macro_key"] == "cirugias"]["denominador"].sum()
else:
    total_cirugias = 0
    meta_cirugias = 0

if unidades_consultas > 0:
    total_consultas = df_plot[df_plot["macro_key"] == "consultas"]["numerador"].sum()
    meta_consultas = df_plot[df_plot["macro_key"] == "consultas"]["denominador"].sum()
else:
    total_consultas = 0
    meta_consultas = 0

# Promedios de avance
avg_avance_general = df_plot["pct_avance_meta_general"].mean()
avg_avance_cirugias = df_plot[df_plot["macro_key"] == "cirugias"]["pct_avance_meta_general"].mean() if unidades_cirugias > 0 else 0
avg_avance_consultas = df_plot[df_plot["macro_key"] == "consultas"]["pct_avance_meta_general"].mean() if unidades_consultas > 0 else 0

# ============================================================
# 5) CAPAS DEL MAPA
# ============================================================

spike_layer = pdk.Layer(
    "ColumnLayer",
    data=df_plot,
    get_position=["longitud", "latitud"],
    get_elevation="volumen_para_columna * 2",
    radius=1200,
    get_fill_color="color_rgb",
    get_line_color=[0, 0, 0, 0],
    pickable=True,
    auto_highlight=True,
)

# Vista del mapa
view_state = pdk.ViewState(
    latitude=df_plot["latitud"].mean(),
    longitude=df_plot["longitud"].mean(),
    zoom=5,
    pitch=55,
)

# Tooltip simple (sin enlace)
tooltip_text = (
    "{nombre_de_la_unidad}\n"
    "CLUES: {clues_imb}\n"
    "Entidad: {entidad}\n"
    "Nivel: {nivel_atencion}\n"
    "Indicador: {tipo_indicador}\n"
    "Realizado: {numerador_fmt}\n"
    "Meta: {denominador_fmt}\n"
    "Avance: {pct_fmt} ({estado_semaforo})"
)

r = pdk.Deck(
    layers=[spike_layer],
    initial_view_state=view_state,
    tooltip={"text": tooltip_text},
    map_style="dark",
)

# ============================================================
# 6) DASHBOARD HTML (CON POPUP/BOTON DE ENLACE)
# ============================================================

html_map = r.to_html(as_string=True)

# Leyenda del semaforo
leyenda_semaforo_html = f"""
<div style="margin-bottom: 12px;">
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 20px; height: 20px; background-color: {semaforo_colores['rojo']}; border-radius: 3px; margin-right: 10px;"></div>
        <span style="color: #e0e0e0; font-size: 13px;">Rojo: Menos del 50%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 20px; height: 20px; background-color: {semaforo_colores['amarillo']}; border-radius: 3px; margin-right: 10px;"></div>
        <span style="color: #e0e0e0; font-size: 13px;">Amarillo: 50% - 79%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 20px; height: 20px; background-color: {semaforo_colores['verde claro']}; border-radius: 3px; margin-right: 10px;"></div>
        <span style="color: #e0e0e0; font-size: 13px;">Verde Claro: 80% - 99%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 20px; height: 20px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 3px; margin-right: 10px;"></div>
        <span style="color: #e0e0e0; font-size: 13px;">Verde Fuerte: 100% o mas</span>
    </div>
</div>
"""

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Dashboard de Productividad por Unidad</title>
<style>
    * {{
        margin: 0;
        padding: 0;
        box-sizing: border-box;
    }}
    
    body {{
        margin: 0;
        font-family: 'Segoe UI', Arial, sans-serif;
        background: #0a0a0a;
        overflow: hidden;
    }}
    
    .header {{
        background: linear-gradient(135deg, rgba(0,0,0,0.8) 0%, rgba(0,0,0,0.6) 100%);
        color: #e0e0e0;
        padding: 20px 30px;
        font-size: 24px;
        font-weight: bold;
        text-shadow: 2px 2px 8px rgba(0,0,0,0.5);
        letter-spacing: 1px;
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        z-index: 10;
        backdrop-filter: blur(10px);
        border-bottom: 2px solid rgba(224,224,224,0.2);
    }}
    
    .kpis {{
        display: grid;
        grid-template-columns: repeat(4, 1fr);
        gap: 15px;
        padding: 20px 25px;
        position: absolute;
        top: 80px;
        left: 0;
        right: 280px;
        z-index: 10;
    }}
    
    .card {{
        background: rgba(0,0,0,0.6);
        backdrop-filter: blur(8px);
        padding: 15px 20px;
        border-radius: 12px;
        transition: all 0.3s ease;
        border: 1px solid rgba(224,224,224,0.3);
    }}
    
    .card:hover {{
        transform: translateY(-3px);
        border-color: rgba(78,205,196,0.5);
        background: rgba(0,0,0,0.75);
    }}
    
    .card-title {{
        font-size: 12px;
        color: #b0b0b0;
        text-transform: uppercase;
        letter-spacing: 1.5px;
        margin-bottom: 8px;
        font-weight: 600;
    }}
    
    .card-value {{
        font-size: 32px;
        font-weight: bold;
        color: white;
        text-shadow: 1px 1px 3px rgba(0,0,0,0.5);
        line-height: 1.1;
    }}
    
    .card-sub {{
        font-size: 11px;
        color: #a0a0a0;
        margin-top: 5px;
    }}
    
    .legend {{
        position: absolute;
        bottom: 20px;
        right: 20px;
        background: rgba(0,0,0,0.75);
        backdrop-filter: blur(10px);
        padding: 15px 20px;
        border-radius: 12px;
        border: 1px solid rgba(224,224,224,0.2);
        z-index: 10;
        min-width: 200px;
    }}
    
    .legend-title {{
        color: white;
        font-size: 14px;
        font-weight: bold;
        margin-bottom: 12px;
        text-align: center;
        letter-spacing: 1px;
        border-bottom: 1px solid rgba(224,224,224,0.3);
        padding-bottom: 8px;
    }}
    
    .stats {{
        position: absolute;
        bottom: 20px;
        left: 20px;
        background: rgba(0,0,0,0.6);
        backdrop-filter: blur(8px);
        padding: 12px 18px;
        border-radius: 10px;
        font-size: 11px;
        color: #ccc;
        z-index: 10;
        border: 1px solid rgba(224,224,224,0.2);
    }}
    
    /* Estilos del boton flotante */
    .floating-btn {{
        position: absolute;
        top: 100px;
        right: 20px;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 12px 24px;
        border-radius: 50px;
        text-decoration: none;
        font-weight: bold;
        font-size: 14px;
        z-index: 20;
        box-shadow: 0 4px 15px rgba(0,0,0,0.3);
        transition: all 0.3s ease;
        display: flex;
        align-items: center;
        gap: 8px;
        border: none;
        cursor: pointer;
        font-family: 'Segoe UI', Arial, sans-serif;
    }}
    
    .floating-btn:hover {{
        transform: translateY(-2px);
        box-shadow: 0 6px 20px rgba(0,0,0,0.4);
        background: linear-gradient(135deg, #764ba2 0%, #667eea 100%);
    }}
    
    /* Modal/Popup */
    .modal {{
        display: none;
        position: fixed;
        z-index: 1000;
        left: 0;
        top: 0;
        width: 100%;
        height: 100%;
        background-color: rgba(0,0,0,0.5);
        backdrop-filter: blur(5px);
    }}
    
    .modal-content {{
        background: linear-gradient(135deg, #1e1e2f 0%, #2a2a3b 100%);
        margin: 15% auto;
        padding: 30px;
        border-radius: 20px;
        width: 400px;
        text-align: center;
        box-shadow: 0 10px 40px rgba(0,0,0,0.5);
        border: 1px solid rgba(255,255,255,0.2);
        animation: slideIn 0.3s ease;
    }}
    
    @keyframes slideIn {{
        from {{
            transform: translateY(-50px);
            opacity: 0;
        }}
        to {{
            transform: translateY(0);
            opacity: 1;
        }}
    }}
    
    .modal-content h3 {{
        color: white;
        margin-bottom: 20px;
        font-size: 24px;
    }}
    
    .modal-content p {{
        color: #ccc;
        margin-bottom: 25px;
        font-size: 16px;
    }}
    
    .modal-link {{
        display: inline-block;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 12px 30px;
        border-radius: 50px;
        text-decoration: none;
        font-weight: bold;
        margin: 10px;
        transition: all 0.3s ease;
    }}
    
    .modal-link:hover {{
        transform: scale(1.05);
        box-shadow: 0 5px 20px rgba(102,126,234,0.4);
    }}
    
    .close-btn {{
        background: rgba(255,255,255,0.2);
        color: white;
        padding: 10px 25px;
        border-radius: 50px;
        text-decoration: none;
        display: inline-block;
        margin: 10px;
        cursor: pointer;
        transition: all 0.3s ease;
    }}
    
    .close-btn:hover {{
        background: rgba(255,255,255,0.3);
    }}
    
    .main {{
        display: flex;
        height: 100vh;
        width: 100vw;
    }}
    
    .map-container {{
        flex: 1;
        position: relative;
    }}
    
    iframe {{
        width: 100%;
        height: 100%;
        border: none;
    }}
</style>
</head>
<body>

<div class="header">
Hospital Dashboard - Productividad por Unidad Medica
</div>

<div class="kpis">
    <div class="card">
        <div class="card-title">Unidades Activas</div>
        <div class="card-value">{len(df_plot):,}</div>
        <div class="card-sub">Cirugia: {unidades_cirugias} | Consulta: {unidades_consultas}</div>
    </div>
    
    <div class="card">
        <div class="card-title">Avance General</div>
        <div class="card-value">{avg_avance_general:.1f}%</div>
        <div class="card-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    
    <div class="card">
        <div class="card-title">Total Cirugias</div>
        <div class="card-value">{total_cirugias:,.0f}</div>
        <div class="card-sub">Meta: {meta_cirugias:,.0f}</div>
    </div>
    
    <div class="card">
        <div class="card-title">Total Consultas</div>
        <div class="card-value">{total_consultas:,.0f}</div>
        <div class="card-sub">Meta: {meta_consultas:,.0f}</div>
    </div>
</div>

<!-- Boton flotante que abre el popup -->
<button class="floating-btn" onclick="abrirPopup()">
     Ver Dashboard de Reportes
</button>

<!-- Modal/Popup -->
<div id="miPopup" class="modal">
    <div class="modal-content">
        <h3> Dashboard de Reportes</h3>
        <p>Accede al dashboard completo para visualizar reportes detallados y análisis avanzados</p>
        <a href="https://argontc.shinyapps.io/pptx/" target="_blank" class="modal-link">
            Abrir Dashboard →
        </a>
        <br/>
        <span class="close-btn" onclick="cerrarPopup()">Cerrar</span>
    </div>
</div>

<div class="legend">
    <div class="legend-title">Semaforo de Avance de Metas</div>
    {leyenda_semaforo_html}
    <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid rgba(224,224,224,0.2); font-size: 10px; color: #888; text-align: center;">
        Altura de barra = Volumen realizado
    </div>
</div>

<div class="stats">
    {df_plot['entidad'].nunique()} Entidades | {len(df_plot)} Unidades georreferenciadas
</div>

<div class="main">
    <div class="map-container">
        <iframe srcdoc='{html_map.replace("'", "&apos;")}'></iframe>
    </div>
</div>

<script>
    function abrirPopup() {{
        document.getElementById('miPopup').style.display = 'block';
    }}
    
    function cerrarPopup() {{
        document.getElementById('miPopup').style.display = 'none';
    }}
    
    // Cerrar popup si se hace clic fuera del contenido
    window.onclick = function(event) {{
        var modal = document.getElementById('miPopup');
        if (event.target == modal) {{
            modal.style.display = 'none';
        }}
    }}
</script>

</body>
</html>
"""

# ============================================================
# 7) GUARDAR Y ABRIR
# ============================================================

output_path = Path.home() / "dashboard_productividad_unidades.html"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

webbrowser.open(str(output_path))

True

In [12]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)